In [1]:
!pip install pyspark

In [2]:
import pyspark
print(pyspark.__version__)

4.0.2


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Celebal Assignment") \
    .getOrCreate()


In [5]:
import random
from datetime import datetime, timedelta

regions = ["West", "East", "North", "South"]
categories = ["Electronics", "Clothing", "Furniture", "Grocery"]
cities = ["Los Angeles", "Seattle", "New York", "Chicago", "Houston"]
subscriptions = ["Premium", "Basic"]
statuses = ["Completed", "Pending", "Cancelled"]

data = []

start_date = datetime(2024, 1, 1)

for i in range(100):

    user_id = random.randint(1000, 1100)

    transaction_date = (
        start_date + timedelta(days=random.randint(0, 60))
    ).strftime("%Y-%m-%d")

    region = random.choice(regions)

    product_category = random.choice(categories)

    sale_amount = round(random.uniform(100, 5000), 2)

    city = random.choice(cities)

    age = random.randint(18, 60)

    subscription = random.choice(subscriptions)

    raw_timestamp = (
        start_date + timedelta(
            days=random.randint(0, 60),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59)
        )
    ).strftime("%Y-%m-%d %H:%M:%S")

    # Introduce some null emails
    email = f"user{i}@gmail.com" if random.random() > 0.08 else None

    # Introduce some empty usernames
    username = f"user{i}" if random.random() > 0.05 else ""

    # Introduce some null prices
    price = round(random.uniform(50, 3000), 2) if random.random() > 0.07 else None

    # Introduce some null statuses
    status = random.choice(statuses) if random.random() > 0.07 else None

    store_id = random.randint(1, 5)

    data.append((
        user_id,
        transaction_date,
        region,
        product_category,
        sale_amount,
        city,
        age,
        subscription,
        raw_timestamp,
        email,
        username,
        price,
        status,
        store_id
    ))

# Add 12 duplicate rows
for i in range(12):
    data.append(data[random.randint(0, 99)])

In [6]:
columns = [
    "user_id",
    "transaction_date",
    "region",
    "product_category",
    "sale_amount",
    "city",
    "age",
    "subscription",
    "raw_timestamp",
    "email",
    "username",
    "price",
    "status",
    "store_id"
]

df = spark.createDataFrame(data, columns)

df.show(10, truncate=False)

print("Total rows:", df.count())

+-------+----------------+------+----------------+-----------+-----------+---+------------+-------------------+---------------+--------+-------+---------+--------+
|user_id|transaction_date|region|product_category|sale_amount|city       |age|subscription|raw_timestamp      |email          |username|price  |status   |store_id|
+-------+----------------+------+----------------+-----------+-----------+---+------------+-------------------+---------------+--------+-------+---------+--------+
|1095   |2024-01-04      |West  |Clothing        |1867.11    |Los Angeles|20 |Basic       |2024-02-21 10:32:00|user0@gmail.com|user0   |487.16 |Completed|2       |
|1072   |2024-02-18      |West  |Clothing        |124.83     |Seattle    |25 |Premium     |2024-02-03 18:27:00|user1@gmail.com|user1   |2374.96|Pending  |5       |
|1038   |2024-01-28      |South |Furniture       |2652.2     |Chicago    |40 |Basic       |2024-02-28 10:52:00|user2@gmail.com|user2   |870.31 |Cancelled|1       |
|1022   |2024-01

In [7]:
df.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- store_id: long (nullable = true)



In [8]:
df.describe().show()

+-------+------------------+----------------+------+----------------+------------------+-------+------------------+------------+-------------------+---------------+--------+------------------+---------+------------------+
|summary|           user_id|transaction_date|region|product_category|       sale_amount|   city|               age|subscription|      raw_timestamp|          email|username|             price|   status|          store_id|
+-------+------------------+----------------+------+----------------+------------------+-------+------------------+------------+-------------------+---------------+--------+------------------+---------+------------------+
|  count|               112|             112|   112|             112|               112|    112|               112|         112|                112|            106|     112|               109|      108|               112|
|   mean|1045.2142857142858|            NULL|  NULL|            NULL| 2438.400178571428|   NULL|              37

In [9]:
print("Total rows:", df.count())

Total rows: 112


In [10]:
print("Total rows:", len(df.columns))

Total rows: 14


In [11]:
print("Rows before removing duplicates:", df.count())

Rows before removing duplicates: 112


In [12]:
df_duplicates_deleted = df.dropDuplicates(["user_id", "transaction_date"])
print("Rows after removing duplicates:", df_duplicates_deleted.count())

Rows after removing duplicates: 100


In [13]:
from pyspark.sql.functions import col, count, when

In [15]:
df_duplicates_deleted.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_duplicates_deleted.columns
]).show()

+-------+----------------+------+----------------+-----------+----+---+------------+-------------+-----+--------+-----+------+--------+
|user_id|transaction_date|region|product_category|sale_amount|city|age|subscription|raw_timestamp|email|username|price|status|store_id|
+-------+----------------+------+----------------+-----------+----+---+------------+-------------+-----+--------+-----+------+--------+
|      0|               0|     0|               0|          0|   0|  0|           0|            0|    5|       0|    3|     3|       0|
+-------+----------------+------+----------------+-----------+----+---+------------+-------------+-----+--------+-----+------+--------+



In [16]:
df_clean = df_duplicates_deleted.na.fill({"price":0})
df_clean = df_clean.na.fill({"status":"Unknown"})

In [17]:
from pyspark.sql.functions import trim

df_clean = df_clean.filter(
    col("email").isNotNull() &
    (trim(col("username")) != "")
)

In [18]:
print("Rows after cleaning:", df_clean.count())

Rows after cleaning: 91


In [19]:
df_clean.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = false)
 |-- status: string (nullable = false)
 |-- store_id: long (nullable = true)



In [20]:
from pyspark.sql.types import TimestampType

df_clean = (
    df_clean
    .withColumn(
        "event_time",
        col("raw_timestamp").cast(TimestampType())
    )
    .drop("raw_timestamp")
)

In [21]:
df_clean.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = false)
 |-- status: string (nullable = false)
 |-- store_id: long (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [22]:
df_filtered = df_clean.filter(
    (col("age").between(18,30)) &
    (col("subscription")=="Premium")
)

df_filtered.show()

+-------+----------------+------+----------------+-----------+-----------+---+------------+----------------+--------+-------+---------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|       city|age|subscription|           email|username|  price|   status|store_id|         event_time|
+-------+----------------+------+----------------+-----------+-----------+---+------------+----------------+--------+-------+---------+--------+-------------------+
|   1002|      2024-01-28| North|        Clothing|     4266.0|Los Angeles| 24|     Premium|user18@gmail.com|  user18|2076.58|Cancelled|       3|2024-02-13 06:55:00|
|   1004|      2024-02-23|  East|        Clothing|    2283.17|   New York| 24|     Premium|user29@gmail.com|  user29| 587.88|  Pending|       3|2024-01-28 13:49:00|
|   1017|      2024-02-03|  East|     Electronics|    3915.06|    Chicago| 27|     Premium|user59@gmail.com|  user59|2409.01|Cancelled|       5|2024-01-13 04:16:00|
|   1022| 

In [23]:
from pyspark.sql.functions import sum, avg, min, max

In [24]:
df_clean.agg(
    sum("price").alias("Total Price"),
    avg("price").alias("Average Price"),
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price")
).show()

+------------------+------------------+-------------+-------------+
|       Total Price|     Average Price|Minimum Price|Maximum Price|
+------------------+------------------+-------------+-------------+
|123627.48000000001|1358.5437362637365|          0.0|      2970.45|
+------------------+------------------+-------------+-------------+



In [25]:
df_clean.groupBy("product_category") \
    .agg(
        sum("sale_amount").alias("Revenue"),
        avg("sale_amount").alias("Average Sales")
    ) \
    .show()

+----------------+------------------+------------------+
|product_category|           Revenue|     Average Sales|
+----------------+------------------+------------------+
|         Grocery|          59371.19|2473.7995833333334|
|     Electronics|59727.090000000004|2714.8677272727273|
|        Clothing|          47772.57| 2274.884285714286|
|       Furniture|          43972.96|1832.2066666666667|
+----------------+------------------+------------------+



In [26]:
df_clean.groupBy("store_id") \
    .agg(
        sum("sale_amount").alias("Total Revenue")
    ) \
    .show()

+--------+------------------+
|store_id|     Total Revenue|
+--------+------------------+
|       5|34018.090000000004|
|       1|          36660.29|
|       3|          51058.84|
|       2| 50539.09000000001|
|       4|           38567.5|
+--------+------------------+



In [27]:
df_clean.filter(
    (col("region") == "West") &
    (col("product_category") == "Electronics")
).show()

+-------+----------------+------+----------------+-----------+-----------+---+------------+----------------+--------+-------+---------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|       city|age|subscription|           email|username|  price|   status|store_id|         event_time|
+-------+----------------+------+----------------+-----------+-----------+---+------------+----------------+--------+-------+---------+--------+-------------------+
|   1002|      2024-01-12|  West|     Electronics|    4205.78|Los Angeles| 50|       Basic|user53@gmail.com|  user53|1222.49|Completed|       4|2024-02-23 16:39:00|
|   1017|      2024-02-04|  West|     Electronics|    3741.58|    Chicago| 35|       Basic|user66@gmail.com|  user66|2799.04|Cancelled|       2|2024-02-06 19:38:00|
|   1030|      2024-02-14|  West|     Electronics|    4858.42|    Houston| 44|     Premium|user95@gmail.com|  user95| 817.74|Completed|       3|2024-02-04 06:10:00|
+-------+-

In [28]:
from pyspark.sql.functions import count

df_clean.groupBy("city") \
    .agg(count("*").alias("record_count")) \
    .filter(col("record_count") > 10) \
    .show()

+-----------+------------+
|       city|record_count|
+-----------+------------+
|Los Angeles|          16|
|    Chicago|          16|
|    Seattle|          18|
|    Houston|          22|
|   New York|          19|
+-----------+------------+



In [29]:
df_clean = df_clean.withColumnRenamed(
    "product_category",
    "category"
)

In [30]:
df_clean.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = false)
 |-- status: string (nullable = false)
 |-- store_id: long (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [31]:
temp_df = df_clean.drop("status")


In [32]:
df_clean.columns

['user_id',
 'transaction_date',
 'region',
 'category',
 'sale_amount',
 'city',
 'age',
 'subscription',
 'email',
 'username',
 'price',
 'status',
 'store_id',
 'event_time']

In [33]:
temp_df.columns

['user_id',
 'transaction_date',
 'region',
 'category',
 'sale_amount',
 'city',
 'age',
 'subscription',
 'email',
 'username',
 'price',
 'store_id',
 'event_time']

In [34]:
df_clean.groupBy("region") \
    .agg(sum("sale_amount").alias("Revenue")) \
    .show()

+------+-----------------+
|region|          Revenue|
+------+-----------------+
| South|66138.46999999999|
|  East|48756.78999999999|
|  West|          48919.3|
| North|         47029.25|
+------+-----------------+



In [35]:
final_df = (
    df.dropDuplicates(["user_id","transaction_date"])
      .na.fill({"price":0})
      .na.fill({"status":"Unknown"})
      .filter(
          col("email").isNotNull() &
          (trim(col("username")) != "")
      )
)

revenue_df = (
    final_df.groupBy("store_id")
            .agg(sum("sale_amount").alias("total_revenue"))
)

revenue_df.show()

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|       5|34018.090000000004|
|       1|          36660.29|
|       3|          51058.84|
|       2| 50539.09000000001|
|       4|           38567.5|
+--------+------------------+

